<h1>Frequency Domain Steganography</h1>

### Necessary Imports & Declarations

In [1]:
import os
import glob
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import random
import pandas as pd
import gc
import pickle
#matplotlib inline
from io import StringIO
from contextlib import redirect_stdout

from PIL import Image
from torchvision import transforms
from skimage.filters import threshold_otsu

from deepgaze.saliency_map import FasaSaliencyMapping

# Define the directory paths
train_dir = os.path.join("/home/btm0050/Research_AI_Stegno_Analysis/SteganoGan/mount/ResearchDataset/originals")

LOG_DIR = "/home/btm0050/Research_AI_Stegno_Analysis/SteganoGan/100bitlogs"
SAL_OUTPUT_DIR = "/home/btm0050/Research_AI_Stegno_Analysis/SteganoGan/100bitlogs/saliency_maps"
CHART_OUTPUT_DIR = "/home/btm0050/Research_AI_Stegno_Analysis/SteganoGan/100bitlogs/charts"
#Add Log Files
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(SAL_OUTPUT_DIR, exist_ok=True)
os.makedirs(CHART_OUTPUT_DIR, exist_ok=True)

#GLOBAL CONSTANTS FOR BATCH PROCESSING
BATCH_INDEX = globals().get("BATCH_INDEX", 0)

###Logging FN

In [2]:
def log_to_file(text, file_name):
    full_path = os.path.join(LOG_DIR, file_name)
    with open(full_path, "a") as f:
        f.write(text + "\n")

### Preview function to see output of each step.

In [3]:
# train_dir = os.path.join("data", "imagenet", "train")
# TAG: run_after_first
train_dir = os.path.join("/home/btm0050/Research_AI_Stegno_Analysis/SteganoGan/mount/ResearchDataset/originals")
all_image_paths = sorted(list(set(glob.glob(os.path.join(train_dir, '*.*')))))

# Limit to first 20,000 images only
batch_paths = all_image_paths[BATCH_INDEX * 1000 : (BATCH_INDEX + 1) * 1000]

# Batch size
BATCH_SIZE = 1000

print(f"[INFO] Batch {BATCH_INDEX + 1}: Loaded {len(batch_paths)} images")
print("[DEBUG] train_dir:", train_dir)
#print("[DEBUG] Files found:", glob.glob(os.path.join(train_dir, '*.*')))


[INFO] Batch 1: Loaded 1000 images
[DEBUG] train_dir: /home/btm0050/Research_AI_Stegno_Analysis/SteganoGan/mount/ResearchDataset/originals


### Load and Preprocess Images

In [ ]:
# Load each image, convert to RGB, resize to 224x224, and normalize to [0,1]
from PIL import Image
import numpy as np

sample_images = []
titles = []
current_paths = []  # Add this line to track paths

for path in batch_paths:
    try:
        img = Image.open(path).convert("RGB").resize((224, 224))
        sample_images.append(np.array(img) / 255.0)
        titles.append(os.path.basename(path))
        current_paths.append(path)  # Add this line to track paths
    except Exception as e:
        print(f"[ERROR] Could not load {path}: {e}")

# Display the resized original images horizontally
#fig, axes = plt.subplots(1, len(sample_images), figsize=(len(sample_images) * 4, 4))
#for ax, img, title in zip(axes, sample_images, titles):
#    ax.imshow(img)
#    ax.set_title(title)
#    ax.axis('off')
#plt.tight_layout()
#plt.show()

In [ ]:
def is_low_variance(image, std_threshold=10):
    image_uint8 = (image * 255).astype(np.uint8) if image.max() <= 1.0 else image.astype(np.uint8)
    gray = cv2.cvtColor(image_uint8, cv2.COLOR_BGR2GRAY)
    std = np.std(gray)
    return std < std_threshold  # Low std ⇒ flat image


def is_mostly_white(image, white_pixel_ratio=0.95):
    image_uint8 = (image * 255).astype(np.uint8) if image.max() <= 1.0 else image.astype(np.uint8)
    gray = cv2.cvtColor(image_uint8, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 250, 255, cv2.THRESH_BINARY)
    white_ratio = np.sum(thresh == 255) / thresh.size
    return white_ratio >= white_pixel_ratio  # Mostly white ⇒ bad

def has_enough_edges(image, edge_threshold=100):
    image_uint8 = (image * 255).astype(np.uint8) if image.max() <= 1.0 else image.astype(np.uint8)
    gray = cv2.cvtColor(image_uint8, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    edge_count = np.count_nonzero(edges)
    return edge_count > edge_threshold

def has_full_dynamic_range(image):
    image_uint8 = (image * 255).astype(np.uint8) if image.max() <= 1.0 else image.astype(np.uint8)
    min_val = image_uint8.min()
    max_val = image_uint8.max()
    return min_val == 0 and max_val == 255

def is_useless_image(image):
    return (
        is_low_variance(image, std_threshold=10) or
        is_mostly_white(image, white_pixel_ratio=0.95) or
        not has_enough_edges(image, edge_threshold=100) or
        not has_full_dynamic_range(image)
    )


# Filter out images that are mostly useless
filtered_images = []
filtered_titles = []
filtered_paths = []  # Add this line to track paths
for img, title, path in zip(sample_images, titles, current_paths):  # Update this line
    if is_useless_image(img):
        print(f"[INFO] Excluded {title} due to criteria")
    else:
        filtered_images.append(img)
        filtered_titles.append(title)
        filtered_paths.append(path)  # Add this line to track paths
# Count the number of filtered images
print(f"[INFO] Filtered images: {len(filtered_images)}")
# Count remaining bad images
print(f"[INFO] Bad images: {len(sample_images) - len(filtered_images)}")

sample_images = filtered_images
titles = filtered_titles
current_paths = filtered_paths  # Add this line to update path tracking

#display the bad images
# fig, axes = plt.subplots(1, len(sample_images) - len(filtered_images), figsize=(len(sample_images) * 4, 4))
# for ax, img, title in zip(axes, sample_images, titles):
#     if is_useless_image(img):
#         ax.imshow(img)
#         ax.set_title(title)
#         ax.axis('off')
# plt.tight_layout()
# plt.show()


### DeepGaze Saliency Mapping

In [ ]:
def preprocess_for_saliency(img, idx=None):
    if not isinstance(img, np.ndarray):
        raise ValueError(f"[ERROR] Image at index {idx} is not a valid numpy array")

    if img.ndim != 3 or img.shape[2] != 3:
        raise ValueError(f"[ERROR] Image at index {idx} must be RGB with shape (H, W, 3), got shape {img.shape}")

    if np.any(np.isnan(img)) or np.any(np.isinf(img)):
        raise ValueError(f"[ERROR] Image at index {idx} contains NaNs or Infs")

    if img.shape != (224, 224, 3):
        raise ValueError(f"[ERROR] Image at index {idx} has unexpected size {img.shape}, expected (224, 224, 3)")

    if img.dtype != np.uint8:
        img = np.clip(img * 255, 0, 255).astype(np.uint8)

    # NEW: Reject images with uniform pixel values across all pixels
    if np.all(img == img[0, 0, :]):
        raise ValueError(f"[ERROR] Image at index {idx} has uniform pixel values: {img[0,0,:]}")

    return img

def gen_saliency_maps(images):
    saliency_maps = []
    valid_indices = []

    for idx, img in enumerate(images):
        try:
            image_uint8 = preprocess_for_saliency(img, idx)

            # Extra validation before saliency call
            if image_uint8.shape != (224, 224, 3):
                raise ValueError(f"Image {idx+1} has invalid shape: {image_uint8.shape}")
            if image_uint8.dtype != np.uint8:
                raise ValueError(f"Image {idx+1} is not uint8")

            if np.any(image_uint8 < 0) or np.any(image_uint8 > 255):
                raise ValueError(f"Image {idx+1} has pixel values out of 0-255 range")

            saliency = FasaSaliencyMapping(image_h=224, image_w=224)
            print(f"✅ Processing image {idx+1}: shape {image_uint8.shape}, min {image_uint8.min()}, max {image_uint8.max()}")
            sal_map = saliency.returnMask(image_uint8, tot_bins=8, format='RGB')
            saliency_maps.append(sal_map)
            valid_indices.append(idx)
        except Exception as e:
            print(f"❌ Skipping image {idx+1} ({titles[idx]}): {e}")
            with open("100bitlogs/skipped_saliency_images.txt", "a", encoding="utf-8") as logf:
                logf.write(f"{titles[idx]}: {e}\n")

    return np.array(saliency_maps), valid_indices
# Generate saliency maps for our sample images
# Generate saliency maps for valid images
saliency_maps, valid_indices = gen_saliency_maps(sample_images)

# Filter other lists to match the successful images only
sample_images = [sample_images[i] for i in valid_indices]
titles = [titles[i] for i in valid_indices]
current_paths = [current_paths[i] for i in valid_indices]  # Add this line to update path tracking

# For each saliency map, create a binary mask using Otsu's thresholding.
binary_masks = []
for sal_map in saliency_maps:
    # Normalize saliency map to [0,255] for thresholding
    sal_map_norm = cv2.normalize(sal_map, None, 0, 255, cv2.NORM_MINMAX).astype('uint8')
    thresh_val = threshold_otsu(sal_map_norm)
    binary_mask = (sal_map_norm < thresh_val).astype(np.uint8) * 255
    binary_masks.append(binary_mask)


<h2>Frequency Domain Embedding (DCT)</h2>

### Gnereate the block mask

In [ ]:
def compute_block_mask(saliency_map, block_size=16, threshold=0.05):
    """
    Compute a binary mask indicating which blocks are good for embedding
    based on average saliency per block.

    Args:
        saliency_map (numpy array): Grayscale saliency map (HxW), normalized [0,1]
        block_size (int): Block size, default 16
        threshold (float): Saliency threshold

    Returns:
        mask (numpy array): Binary mask (h_blocks x w_blocks), 1=good for embedding, 0=skip
    """
    h, w = saliency_map.shape
    h_blocks = h // block_size
    w_blocks = w // block_size

    mask = np.zeros((h_blocks, w_blocks), dtype=np.uint8)

    for i in range(h_blocks):
        for j in range(w_blocks):
            y_start = i * block_size
            x_start = j * block_size
            block = saliency_map[y_start:y_start + block_size, x_start:x_start + block_size]
            avg_saliency = np.mean(block)

            if avg_saliency < threshold:
                mask[i, j] = 1  # Good block for embedding

    return mask


### Overlay the Mask on the Image

In [ ]:
def overlay_blocks_on_image(image, block_mask, block_size=16):
    """
    Overlay green/red rectangles on the original image based on the block mask.

    Args:
        image (numpy array): Original RGB image (HxWx3), uint8
        block_mask (numpy array): Binary block mask (h_blocks x w_blocks)
        block_size (int): Block size, default 16

    Returns:
        overlay_img (numpy array): Image with green/red block overlays
    """
    overlay_img = image.copy()
    h_blocks, w_blocks = block_mask.shape

    for i in range(h_blocks):
        for j in range(w_blocks):
            y_start = i * block_size
            x_start = j * block_size

            if block_mask[i, j] == 1:
                color = (0, 255, 0)  # Green for good block
            else:
                color = (255, 0, 0)  # Red for skip block

            # Draw rectangle
            cv2.rectangle(
                overlay_img,
                (x_start, y_start),
                (x_start + block_size - 1, y_start + block_size - 1),
                color,
                1
            )

    return overlay_img


### Display Saliency Maps and Block Overlays

In [ ]:
import random

def display_saliency_and_overlay(saliency_maps, overlay_images, titles=None):
    """
    Display saliency maps and overlay images in two rows:
    - Row 0: Saliency maps (hot colormap)
    - Row 1: Overlay images with green/red blocks

    Args:
        saliency_maps (list): List of saliency maps (HxW), normalized [0,1] or [0,255]
        overlay_images (list): List of overlay images (HxWx3), uint8
        titles (list): Optional list of titles for each image
    """
    # Randomly select 10 indices
    num_images = len(saliency_maps)
    selected_indices = random.sample(range(num_images), min(10, num_images))

    # Filter the saliency maps, overlay images, and titles based on selected indices
    saliency_maps = [saliency_maps[i] for i in selected_indices]
    overlay_images = [overlay_images[i] for i in selected_indices]
    if titles:
        titles = [titles[i] for i in selected_indices]

    # Display the selected images
    fig, axes = plt.subplots(2, len(saliency_maps), figsize=(len(saliency_maps) * 4, 8))

    for i in range(len(saliency_maps)):
        sal_map = saliency_maps[i]
        overlay_img = overlay_images[i]

        # Normalize saliency map for display
        if sal_map.max() <= 1.0:
            sal_map_vis = (sal_map * 255).astype(np.uint8)
        else:
            sal_map_vis = sal_map.astype(np.uint8)

        # Saliency map
        axes[0, i].imshow(sal_map_vis, cmap='hot')
        axes[0, i].set_title(f"Saliency Map {i+1}" if not titles else titles[i])
        axes[0, i].axis('off')

        # Overlay image
        axes[1, i].imshow(overlay_img)
        axes[1, i].set_title(f"Block Overlay {i+1}" if not titles else titles[i])
        axes[1, i].axis('off')

    plt.tight_layout()
    plt.show()


# Parameters
block_size = 8  # Changed to 8 after 16
threshold = 0.05

# Containers for masks and overlay images
block_masks = []
overlay_images = []

# Process each image and saliency map
for img, sal_map in zip(sample_images, saliency_maps):
    # Ensure image is uint8
    img_uint8 = (img * 255).astype(np.uint8) if img.max() <= 1.0 else img

    # Normalize saliency map if necessary
    sal_map_norm = sal_map / 255.0 if sal_map.max() > 1.0 else sal_map

    # Step 1: Compute block mask
    block_mask = compute_block_mask(sal_map_norm, block_size, threshold)
    block_masks.append(block_mask)

    # Step 2: Overlay block mask on image
    overlay_img = overlay_blocks_on_image(img_uint8, block_mask, block_size)
    overlay_images.append(overlay_img)

# Step 3: Display results for 10 random images
display_saliency_and_overlay(saliency_maps, overlay_images, titles)

# Calculate and display block statistics for all images
for idx, block_mask in enumerate(block_masks):
    total_blocks = block_mask.size
    good_blocks = np.sum(block_mask == 1)
    bad_blocks = total_blocks - good_blocks

    print(f"Image {idx+1}: Total Blocks = {total_blocks}, Good Blocks = {good_blocks}, Bad Blocks = {bad_blocks}")

### Calculate Capacity

In [ ]:
def calculate_embedding_capacity(block_mask, bits_per_block=4):
    """
    Calculate embedding capacity based on block mask and bits per block.
    
    Args:
        block_mask (numpy array): Binary mask of blocks (h_blocks x w_blocks)
        bits_per_block (int): Number of bits you can embed per block (default: 4)
    
    Returns:
        total_bits (int): Total embedding capacity in bits
    """
    num_good_blocks = np.sum(block_mask == 1)
    total_bits = num_good_blocks * bits_per_block
    print(f"Embedding capacity: {total_bits} bits ({num_good_blocks} blocks x {bits_per_block} bits/block)")
    return total_bits



### Generate Random Payload Bits

In [ ]:
def generate_random_payload(num_bits):
    """
    Generate a random payload as a bitstream.
    
    Args:
        num_bits (int): Number of bits to generate
    
    Returns:
        payload_bits (numpy array): Array of 0s and 1s
    """
    payload_bits = np.random.randint(0, 2, size=num_bits, dtype=np.uint8)
    print(f"Generated random payload of {num_bits} bits.")
    return payload_bits


### Convert String Message to Bits

In [ ]:
#Used for 100 bit payload 
def string_to_bits(message, nibble):
    """
    Convert a 12-character ASCII message + 4-bit nibble to a 100-bit numpy array.
    
    Args:
        message (str): 12 ASCII characters (96 bits)
        nibble (int): 4-bit integer (0–15) to complete the 100-bit payload

    Returns:
        np.ndarray: 100-bit array of 0s and 1s
    """
    if len(message) != 12:
        raise ValueError("Message must be exactly 12 ASCII characters for 96 bits.")
    if not (0 <= nibble <= 15):
        raise ValueError("Nibble must be a 4-bit integer (0–15).")
    
    bits = []
    for byte in message.encode("ascii"):
        bits.extend([int(b) for b in format(byte, "08b")])
    
    nibble_bits = [int(b) for b in format(nibble, "04b")]
    bits.extend(nibble_bits)

    if len(bits) != 100:
        raise ValueError(f"Bitstream is {len(bits)} bits, expected 100.")
    
    print(f"✅ Converted '{message}' + nibble {nibble} to 100 bits.")
    return np.array(bits, dtype=np.uint8)




#def string_to_bits(message):
#
#    """
#
#    Convert a string message to a bitstream (list of 0s and 1s).
#
#    
#
#    Args:
#
#        message (str): Text message to convert
#
#    
#
#    Returns:
#
#        payload_bits (numpy array): Bitstream of the message
#
#    """
#
#    byte_array = bytearray(message, 'utf-8')
#
#    bits = []
#
#    for byte in byte_array:
#
#        bits.extend([int(bit) for bit in format(byte, '08b')])
#
#    
#
#    payload_bits = np.array(bits, dtype=np.uint8)
#
#    print(f"Converted message '{message}' to {len(payload_bits)} bits.")
#
#    return payload_bits


###Check Message Length

In [ ]:
# payload100 = string_to_bits("hiddendata42", 0b1010)
# payload200 = string_to_bits_200("Data must remain private", 0b10101010)
# payload300 = string_to_bits_300("Encrypt data safely with strong keys.", 0b1010)
# payload400 = string_to_bits_400("Data privacy is essential for trust and security.", 0b10101010)
#payload500 = string_to_bits_500("Confidentiality, integrity, and authenticity must prevail!", 0b1101)

payload100 = string_to_bits("hiddendata42", 0b1010)
print(f"Payload length: {len(payload100)} bits")
print(f"Payload bits: {payload100}")

In [ ]:
# Assume you already have block_masks from earlier

# Step 1: Calculate capacity for each image
bits_per_block = 4  # Or adjust based on your DCT embedding plan
capacities = [calculate_embedding_capacity(block_mask, bits_per_block=bits_per_block) for block_mask in block_masks]

# Step 2a: Generate random payloads for each image
# random_payloads = [generate_random_payload(capacity) for capacity in capacities]

# Optional Step 2b: Convert string message to bits (if you want)
#message = "War Eagle!"
# The quick brown fox jumps over the lazy dog!
message_payload = payload100 #string_to_bits(message)

# Ensure message payload fits for each image
# payloads = []
# for capacity in capacities:
#     if len(message_payload) > capacity:
#         print("Message is too big! Trimming it to fit.")
#         payload = message_payload[:capacity]
#     else:
#         payload = message_payload
#     payloads.append(payload)

payloads = []
for i, capacity in enumerate(capacities):
    if len(message_payload) <= capacity:
        print(f"Message fits for image {i}, using your message!")
        payload = message_payload
    else:
        print(f"Message too big for image {i}, using random bits instead.")
        payload = generate_random_payload(capacity)
    payloads.append(payload)

for i, payload in enumerate(payloads):
    print(f'Payload for image {i}: {payload}')
# Now you have payload bits ready to embed for each image!


## STEP 6 - Qunatization tables for luminance and chrominance

### Extract JPEG Quantization Tables (QnT)

In [ ]:
def extract_quantization_tables(image_paths):
    """
    Extract JPEG quantization tables from image files.

    Args:
        image_paths (list): List of file paths to images.

    Returns:
        qtables_list (list): List of quantization tables for each image.
    """
    qtables_list = []

    for idx, img_path in enumerate(image_paths):
        try:
            img = Image.open(img_path)

            # Check if it's JPEG
            if img.format != 'JPEG':
                print(f"[{idx+1}] {os.path.basename(img_path)} is not a JPEG image. Skipping.")
                qtables_list.append(None)
                continue

            qtables = img.quantization
            qtables_list.append(qtables)

            print(f"[{idx+1}] Extracted quantization tables from {os.path.basename(img_path)}")

        except Exception as e:
            print(f"[{idx+1}] Error processing {img_path}: {e}")
            qtables_list.append(None)

    return qtables_list

# Run it on your sample image paths
qtables_list = extract_quantization_tables(batch_paths)

Visualize the QnT Tables (for Understanding)

In [ ]:
import random

def display_quantization_tables_sample(qtables_list, image_titles, sample_size=10):
    """
    Display quantization tables for a random sample of images in a readable format.

    Args:
        qtables_list (list): List of quantization tables.
        image_titles (list): Corresponding titles or filenames.
        sample_size (int): Number of images to sample and display.
    """
    # Ensure the lists are not empty
    if not qtables_list or not image_titles:
        print("Error: qtables_list or image_titles is empty.")
        return

    # Ensure the lengths of qtables_list and image_titles match
    if len(qtables_list) != len(image_titles):
        print("Error: Length mismatch between qtables_list and image_titles.")
        return

    # Randomly select indices for sampling
    sampled_indices = random.sample(range(len(qtables_list)), min(sample_size, len(qtables_list)))

    for idx in sampled_indices:
        qtables = qtables_list[idx]
        print(f"\n=== Quantization Tables for Image {idx+1}: {image_titles[idx]} ===")
        
        if qtables is None:
            print("No quantization table extracted.")
            continue

        for table_id, qtable in qtables.items():
            try:
                qtable_matrix = np.array(qtable).reshape((8, 8))
                df = pd.DataFrame(qtable_matrix)
                print(f"\nTable {table_id} ({'Luminance' if table_id == 0 else 'Chrominance'}):")
                display(df)
            except Exception as e:
                print(f"Error processing table {table_id}: {e}")

# Filter the qtables_list to match the sample images using valid_indices
filtered_qtables_list = [qtables_list[i] for i in valid_indices]

# Visualize the quantization tables for a random sample of 10 images
display_quantization_tables_sample(filtered_qtables_list, titles, sample_size=10)


## Step 7 - Convert "Good" 16x16 Blocks into DCT Domain
Take each 16x16 "good" block (from your block_masks)
Split it into four 8x8 sub-blocks (because DCT is done in 8x8 blocks)
Perform 2D DCT on each sub-block
Store those DCT coefficients for embedding in the next step


### -  Code Block #1: Helper Function to Perform 2D DCT

In [ ]:
# def block_to_dct_blocks(block_16x16):
#     """
#     Split a 16x16 block into four 8x8 sub-blocks and perform DCT on each.
    
#     Args:
#         block_16x16 (numpy array): 16x16x3 block (RGB).
    
#     Returns:
#         dct_blocks (list): List of 4 DCT blocks (each 8x8x3).
#     """
#     dct_blocks = []

#     # Split the 16x16 block into four 8x8 blocks
#     for i in range(2):  # rows
#         for j in range(2):  # columns
#             y_start = i * 8
#             x_start = j * 8

#             sub_block = block_16x16[y_start:y_start+8, x_start:x_start+8, :]

#             # Perform DCT on each channel separately
#             dct_sub_block = np.zeros_like(sub_block, dtype=np.float32)
#             for c in range(3):  # R, G, B channels
#                 dct_sub_block[:, :, c] = cv2.dct(sub_block[:, :, c].astype(np.float32))

#             dct_blocks.append(dct_sub_block)

#     return dct_blocks

def block_to_dct_block(block_8x8):
    """
    Perform DCT on an 8x8 RGB block.

    Args:
        block_8x8 (numpy array): 8x8x3 block (RGB).

    Returns:
        dct_block (numpy array): 8x8x3 DCT-transformed block.
    """
    dct_block = np.zeros_like(block_8x8, dtype=np.float32)

    # Perform DCT on each channel
    for c in range(3):  # R, G, B
        dct_block[:, :, c] = cv2.dct(block_8x8[:, :, c].astype(np.float32))

    return dct_block



### - Code Block #2: Apply to All Images and Good Blocks

In [ ]:
def extract_dct_blocks_from_good_blocks_8x8(images, block_masks, block_size=8, log_file="dct_block_log.txt"):
    """
    Extract 8x8 DCT blocks from each 'good' block in all images.

    Args:
        images (list): List of sample images (normalized RGB arrays).
        block_masks (list): List of block masks (1 = good for embedding).
        block_size (int): Size of blocks (now 8x8).
    
    Returns:
        dct_blocks_per_image (list): For each image, a list of DCT blocks.
    """
    log_path = os.path.join(LOG_DIR, log_file)
    dct_blocks_per_image = []

    with open(log_path, "a") as logf:
        logf.write(f"\n=== [Batch {BATCH_INDEX + 1}] Extracting DCT Blocks (Batch Log) ===\n")


        for img_idx, (img, block_mask) in enumerate(zip(images, block_masks)):
            msg1 = f"\nProcessing image {img_idx+1}/{len(images)}..."
            print(msg1)
            logf.write(msg1 + "\n")

            img_uint8 = (img * 255).astype(np.uint8) if img.max() <= 1.0 else img

            h_blocks, w_blocks = block_mask.shape
            dct_blocks_for_this_image = []

            for i in range(h_blocks):
                for j in range(w_blocks):
                    if block_mask[i, j] == 1:
                        y_start = i * block_size
                        x_start = j * block_size

                        block_8x8 = img_uint8[y_start:y_start+block_size, x_start:x_start+block_size, :]
                        dct_block = block_to_dct_block(block_8x8)

                        dct_blocks_for_this_image.append({
                            "block_position": (i, j),
                            "dct_block": dct_block
                        })

            msg2 = f"Extracted {len(dct_blocks_for_this_image)} good 8x8 blocks from image {img_idx+1}"
            print(msg2)
            logf.write(msg2 + "\n")

            dct_blocks_per_image.append(dct_blocks_for_this_image)
        return dct_blocks_per_image

dct_blocks_per_image = extract_dct_blocks_from_good_blocks_8x8(sample_images, block_masks)


In [ ]:
from skimage.measure import shannon_entropy

def extract_block_features(dct_blocks_info, saliency_map, block_size=8):
    """
    Extract features from 8x8 DCT blocks for ML model.

    Args:
        dct_blocks_info (list): DCT block info with 'block_position' and 'dct_block'
        saliency_map (np.array): Normalized saliency map (HxW)
        block_size (int): Block size (default 8)

    Returns:
        features (list): List of feature dicts (one per block)
    """
    features = []

    for block in dct_blocks_info:
        i, j = block["block_position"]
        dct_block = block["dct_block"]

        y_start = i * block_size
        x_start = j * block_size

        # Raw pixel block (just for entropy)
        block_saliency = saliency_map[y_start:y_start+block_size, x_start:x_start+block_size]
        avg_saliency = np.mean(block_saliency)
        entropy = shannon_entropy(block_saliency)
        
        # Variance of DCT coefficients (mid-mid region)
        mid_dct = dct_block[2:6, 2:6, :]  # central region
        dct_var = np.var(mid_dct)

        features.append({
            "block_row": i,
            "block_col": j,
            "avg_saliency": avg_saliency,
            "entropy": entropy,
            "dct_variance": dct_var
        })

    return features

all_features_per_image = []

for idx, dct_blocks in enumerate(dct_blocks_per_image):
    print(f"Extracting features for Image {idx+1}")
    saliency_map = saliency_maps[idx] / 255.0 if saliency_maps[idx].max() > 1.0 else saliency_maps[idx]
    features = extract_block_features(dct_blocks, saliency_map)
    all_features_per_image.append(features)


In [ ]:
def label_features_heuristically(features, saliency_thresh=0.05, variance_thresh=10.0):
    """
    Label each feature row as 1 (good) or 0 (bad) using basic heuristics.

    Args:
        features (list of dicts): Output from extract_block_features()
        saliency_thresh (float): Max saliency value considered "good"
        variance_thresh (float): Min DCT variance considered "good"

    Returns:
        labeled_features (list of dicts): Same as input, with added 'label'
    """
    labeled = []

    for f in features:
        label = int(f["avg_saliency"] < saliency_thresh and f["dct_variance"] > variance_thresh)
        f_labeled = f.copy()
        f_labeled["label"] = label
        labeled.append(f_labeled)

    return labeled

labeled_data = []

for features in all_features_per_image:
    labeled = label_features_heuristically(features)
    labeled_data.extend(labeled)  # flat list for training

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd

# Assuming `labeled_data` is already defined
df = pd.DataFrame(labeled_data)

# Rebalance the dataset (undersample the majority class)
df_1 = df[df["label"] == 1]
df_0 = df[df["label"] == 0]
df_1_downsampled = df_1.sample(len(df_0), random_state=42)
df_balanced = pd.concat([df_0, df_1_downsampled]).sample(frac=1, random_state=42).reset_index(drop=True)

# Feature and label extraction
X = df_balanced[["avg_saliency", "entropy", "dct_variance"]].values
y = df_balanced["label"].values.reshape(-1, 1)

# Convert to tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42)

# Define neural net
class EmbedNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

model = EmbedNet()
model.load_state_dict(torch.load("block_classifier_model.pth", map_location=torch.device('cpu')))
model.eval()

#PRETRAINED
## Metrics storage
#train_losses = []
#val_losses = []
#train_accuracies = []
#val_accuracies = []
#
## Train loop
#for epoch in range(30):
#    model.train()
#    optimizer.zero_grad()
#    outputs = model(X_train)
#    loss = criterion(outputs, y_train)
#    loss.backward()
#    optimizer.step()
#
#    # Training accuracy
#    train_pred = (outputs > 0.6).int()
#    train_acc = (train_pred == y_train.int()).sum().item() / y_train.size(0)
#
#    with torch.no_grad():
#        model.eval()
#        val_outputs = model(X_test)
#        val_loss = criterion(val_outputs, y_test)
#        val_pred = (val_outputs > 0.6).int()
#        val_acc = (val_pred == y_test.int()).sum().item() / y_test.size(0)
#
#    # Save metrics
#    train_losses.append(loss.item())
#    val_losses.append(val_loss.item())
#    train_accuracies.append(train_acc)
#    val_accuracies.append(val_acc)
#
#    print(f"Epoch {epoch+1} — Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}, "
#          f"Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")
#
## Final predictions and metrics
#y_pred = (val_outputs > 0.6).int()
#print("📊 Classification Report:")
#print(classification_report(y_test, y_pred))
#print("🧩 Confusion Matrix:")
#print(confusion_matrix(y_test, y_pred))
#
## Final accuracy
#correct_predictions = (y_pred == y_test.int()).sum().item()
#total_predictions = y_test.size(0)
#accuracy = correct_predictions / total_predictions
#print(f"Accuracy: {accuracy:.4f}")


### Display a chart

import matplotlib.pyplot as plt

# Plot training & validation loss and accuracy over epochs
plt.figure(figsize=(12, 8))

# 📉 Loss Plot
plt.subplot(2, 1, 1)
plt.plot(range(1, len(train_losses) + 1), train_losses, label='Training Loss', marker='o')
plt.plot(range(1, len(val_losses) + 1), val_losses, label='Validation Loss', marker='o')
plt.title('Training and Validation Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# 📈 Accuracy Plot
plt.subplot(2, 1, 2)
plt.plot(range(1, len(train_accuracies) + 1), train_accuracies, label='Training Accuracy', marker='o')
plt.plot(range(1, len(val_accuracies) + 1), val_accuracies, label='Validation Accuracy', marker='o')
plt.title('Training and Validation Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


### Predict the blocks

In [ ]:
def predict_embedding_blocks_strict(model, dct_blocks_info, saliency_map, block_size=8, min_dct_variance=5.0, threshold=0.6):
    """
    Predict good blocks for embedding using a DL model + post-filtering with DCT variance.

    Args:
        model (torch.nn.Module): Trained DL model
        dct_blocks_info (list): List of 8x8 DCT block dicts
        saliency_map (ndarray): Normalized saliency map
        block_size (int): Block size (default 8)
        min_dct_variance (float): Min DCT variance allowed (post-filter)
        threshold (float): DL confidence threshold for embedding

    Returns:
        selected_blocks (list): DL-predicted + variance-checked blocks
    """
    model.eval()
    selected_blocks = []

    for block in dct_blocks_info:
        i, j = block["block_position"]
        dct_block = block["dct_block"]

        y_start = i * block_size
        x_start = j * block_size
        block_saliency = saliency_map[y_start:y_start+block_size, x_start:x_start+block_size]

        avg_sal = np.mean(block_saliency)
        entropy = shannon_entropy(block_saliency)
        dct_var = np.var(dct_block[2:6, 2:6, :])  # Focused frequency band

        # DL Prediction
        features = torch.tensor([[avg_sal, entropy, dct_var]], dtype=torch.float32)
        with torch.no_grad():
            confidence = model(features).item()

        # Post-filter
        if confidence > threshold and dct_var > min_dct_variance:
            selected_blocks.append(block)

    return selected_blocks


ai_selected_blocks_per_image = []

for idx in range(len(dct_blocks_per_image)):
    print(f"AI selecting blocks for Image {idx+1}")

    dct_blocks = dct_blocks_per_image[idx]
    sal_map = saliency_maps[idx] / 255.0 if saliency_maps[idx].max() > 1.0 else saliency_maps[idx]

    selected_blocks = predict_embedding_blocks_strict(model, dct_blocks, sal_map)

    ai_selected_blocks_per_image.append(selected_blocks)

    print(f"Selected {len(selected_blocks)} / {len(dct_blocks)} blocks")


In [ ]:
def draw_ai_selected_blocks_on_images(images, ai_selected_blocks_per_image, block_size=8, color=(0, 255, 0)):
    """
    Overlay AI-selected 8x8 blocks on images.

    Args:
        images (list): List of original images (uint8).
        ai_selected_blocks_per_image (list): List of lists of selected blocks per image.
        block_size (int): Size of each block (default 8).
        color (tuple): BGR color for block outlines (default green).
    
    Returns:
        overlay_images (list): Images with blocks drawn.
    """
    overlay_images = []

    for idx, (image, selected_blocks) in enumerate(zip(images, ai_selected_blocks_per_image)):
        img_copy = image.copy()

        for block in selected_blocks:
            i, j = block["block_position"]
            y_start = i * block_size
            x_start = j * block_size
            cv2.rectangle(img_copy, (x_start, y_start), (x_start + block_size, y_start + block_size), color, 1)

        overlay_images.append(img_copy)

    return overlay_images

# Normalize originals to uint8 if needed
originals_uint8 = [(img * 255).astype(np.uint8) if img.max() <= 1.0 else img for img in sample_images]

ai_overlay_images = draw_ai_selected_blocks_on_images(
    images=originals_uint8,
    ai_selected_blocks_per_image=ai_selected_blocks_per_image
)


In [ ]:
def show_ai_overlay_results(originals, overlays, titles, sample_size=10):
    # Randomly select a sample of images
    num = min(sample_size, len(originals))
    selected_indices = random.sample(range(len(originals)), num)

    # Filter the originals, overlays, and titles based on the selected indices
    sampled_originals = [originals[i] for i in selected_indices]
    sampled_overlays = [overlays[i] for i in selected_indices]
    sampled_titles = [titles[i] for i in selected_indices]

    fig, axes = plt.subplots(2, num, figsize=(num * 4, 8))

    if num == 1:
        axes = np.array(axes).reshape(2, 1)

    for i in range(num):
        axes[0, i].imshow(sampled_originals[i])
        axes[0, i].set_title(f"Original: {sampled_titles[i]}")
        axes[0, i].axis('off')

        axes[1, i].imshow(sampled_overlays[i])
        axes[1, i].set_title(f"AI Selected Blocks: {sampled_titles[i]}")
        axes[1, i].axis('off')

    plt.tight_layout()
    plt.show()

show_ai_overlay_results(originals_uint8, ai_overlay_images, titles)


### Sanity Check & Visualize DCT Coefficients

In [ ]:
def visualize_dct_block(dct_block, title_prefix="DCT Block"):
    """
    Visualize the magnitude spectrum of a single DCT 8x8 block (RGB channels).
    
    Args:
        dct_block (numpy array): 8x8x3 DCT coefficients.
    """
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    for c in range(3):
        # Take absolute value and log scale for visualization
        dct_magnitude = np.log(np.abs(dct_block[:, :, c]) + 1)

        axes[c].imshow(dct_magnitude, cmap='gray')
        axes[c].set_title(f"{title_prefix} - Channel {c} (RGB)")
        axes[c].axis('off')

    plt.tight_layout()
    plt.show()

# Example: Visualize first DCT block of the first image
if dct_blocks_per_image and len(dct_blocks_per_image[0]) > 0:
    # example_dct_block = dct_blocks_per_image[0][0]["dct_block"]  # Directly get the 8x8 DCT block
    if ai_selected_blocks_per_image[0]:
        example_dct_block = ai_selected_blocks_per_image[0][0]["dct_block"]
    visualize_dct_block(example_dct_block, title_prefix="Example DCT Block")
else:
    print("No DCT blocks found to visualize.")



## Step 8: Select Mid-Mid DCT Frequencies for Embedding

- From each **8x8 DCT block** inside your good **16x16 blocks**,  
  we select specific **mid-frequency coefficients** to embed your data.

- We'll **adaptively embed**, considering the **QnT values** we extracted earlier (from Step 6).

---

#### **What We’ll Do**

1. Define a list of **mid-mid frequency positions**  
2. For each DCT block, **select those positions**  
3. Apply **QnT-based decision logic**:
    - Skip **low Q** (highly sensitive)  
    - Allow changes in **mid/high Q**  
4. Prepare to **embed bits** in these coefficients in the next step.


### Code Block #1: Define Mid-Mid Frequencies

In [ ]:
# Mid-mid frequency positions (Y-axis first)
mid_mid_frequencies = [(3, 3), (2, 3), (3, 2), (4, 1), (1, 4)]

### Code Block #2: Adaptive Embedding Decision Function

In [ ]:
def should_embed(q_value, q_threshold_low=4, q_threshold_high=8):
    """
    Decide whether to embed based on quantization value.

    Args:
        q_value (int): Quantization table value at that frequency.
        q_threshold_low (int): Lower threshold (too sensitive to modify).
        q_threshold_high (int): Upper threshold (safe for stronger changes).

    Returns:
        bool: True if we should embed here, False otherwise.
    """
    if q_value <= q_threshold_low:
        return False  # Too sensitive, skip
    elif q_value <= q_threshold_high:
        return True   # Safe for minimal embedding (±1)
    else:
        return True   # Safe for stronger embedding (±1 or ±2)


### Code Block #3: Select Embedding Positions in DCT Blocks


In [ ]:
def select_embedding_positions_8x8(dct_blocks_per_image, qtables_list, mid_mid_frequencies):
    """
    Select positions in 8x8 DCT blocks where we can embed data.

    Args:
        dct_blocks_per_image (list): List of DCT blocks per image (after 8x8 processing).
        qtables_list (list): Quantization tables (one per image).
        mid_mid_frequencies (list): (y, x) tuples indicating DCT frequencies to consider.

    Returns:
        embedding_candidates_per_image (list): For each image, list of embedding positions.
    """
    embedding_candidates_per_image = []

    for img_idx, (dct_blocks_info, qtables) in enumerate(zip(dct_blocks_per_image, qtables_list)):
        print(f"\nSelecting embedding positions for Image {img_idx+1}")

        if qtables is None:
            print("No Q-table found. Skipping.")
            embedding_candidates_per_image.append([])
            continue

        qtable = np.array(qtables[0]).reshape((8, 8))  # Use luminance table
        candidates_for_image = []

        for block_info in dct_blocks_info:
            block_position = block_info['block_position']
            dct_block = block_info['dct_block']  # This is the full 8x8 DCT block

            candidates = []

            for (y, x) in mid_mid_frequencies:
                q_value = qtable[y, x]

                if should_embed(q_value):
                    candidates.append({
                        "block_position": block_position,
                        "freq": (y, x),
                        "q_value": q_value
                    })

            if candidates:
                candidates_for_image.append(candidates)

        print(f"Found {len(candidates_for_image)} blocks with embedding spots in Image {img_idx+1}")
        embedding_candidates_per_image.append(candidates_for_image)

    return embedding_candidates_per_image

embed_log_path = os.path.join(LOG_DIR, "embedding_selection_log.txt")

log_buffer = StringIO()
with redirect_stdout(log_buffer):
    print(f"\n=== [Batch {BATCH_INDEX + 1}] Embedding Position Selection Log ===")
    
    embedding_candidates_per_image = select_embedding_positions_8x8(
        dct_blocks_per_image,
        qtables_list,
        mid_mid_frequencies
    )

# Write log output to file
with open(embed_log_path, "a") as logf:
    logf.write(log_buffer.getvalue())


print(log_buffer.getvalue())


### Code Block #4: Filter Images With Opportunities

In [ ]:
def filter_images_with_candidates(embedding_candidates_per_image, dct_blocks_per_image, image_paths, titles, payloads, bits_per_block=4):  # Rename parameter for clarity
    """
    Filters out images that have no embedding candidates and displays details of remaining images.

    Args:
        embedding_candidates_per_image (list): List of candidates per image.
        dct_blocks_per_image (list): List of DCT block info per image.
        image_paths (list): File paths that have been filtered in sync with other lists.
        titles (list): Titles for each image.
        payloads (list): Payload bits for each image.
        bits_per_block (int): Number of bits that can be embedded per block.

    Returns:
        filtered_candidates (list)
        filtered_dct_blocks (list)
        filtered_image_paths (list)
        filtered_titles (list)
        filtered_payloads (list)
    """
    filtered_candidates = []
    filtered_dct_blocks = []
    filtered_image_paths = []
    filtered_titles = []
    filtered_payloads = []


    for idx, candidates in enumerate(embedding_candidates_per_image):
        if len(candidates) > 0:
            filtered_candidates.append(candidates)
            filtered_dct_blocks.append(dct_blocks_per_image[idx])
            filtered_image_paths.append(image_paths[idx])  # Use filtered paths
            filtered_titles.append(titles[idx])
            filtered_payloads.append(payloads[idx])

    print(f"\nFiltered {len(embedding_candidates_per_image) - len(filtered_candidates)} images with zero embedding opportunities.")
    print(f"Remaining images for embedding: {len(filtered_candidates)}")

    return filtered_candidates, filtered_dct_blocks, filtered_image_paths, filtered_titles, filtered_payloads



# Setup logging
log_dir = "./100bitlogs"
os.makedirs(log_dir, exist_ok=True)
log_file = os.path.join(log_dir, "filtered_candidates_log.txt")
# Capture print output
log_buffer = StringIO()
with redirect_stdout(log_buffer):
    print(f"\n======= BATCH {BATCH_INDEX} =======")
    
    (
        filtered_candidates_per_image,
        filtered_dct_blocks_per_image,
        filtered_sample_image_paths,
        filtered_titles,
        filtered_payloads
    ) = filter_images_with_candidates(
        embedding_candidates_per_image,
        dct_blocks_per_image,
        current_paths,  # Pass the tracked and filtered paths
        titles,
        payloads
    )


### Code Block #4: Sanity Check - Visualize Candidate Counts Per Image

In [ ]:
def visualize_filtered_embedding_candidates(filtered_candidates_per_image, filtered_titles):
    """
    Plot number of embedding candidate blocks for filtered images.

    Args:
        filtered_candidates_per_image (list): Filtered candidate data.
        filtered_titles (list): Titles for remaining images.
    """
    num_blocks = [len(candidates) for candidates in filtered_candidates_per_image]

    plt.figure(figsize=(8, 4))
    plt.bar(range(len(num_blocks)), num_blocks, tick_label=filtered_titles)
    plt.title("Filtered: Embedding Candidate Blocks Per Image")
    plt.xlabel("Image")
    plt.ylabel("Embedding Candidate Blocks")
    plt.xticks(rotation=45)
    plt.show()

# Show filtered candidate block counts
visualize_filtered_embedding_candidates(filtered_candidates_per_image, filtered_titles)



## Step 9: Embed Payload Bits into the Selected DCT Coefficients


#### Step 9 - tasks


- Embed your **payload bits** into the **mid-mid DCT coefficients** of each good block,  
  for each image that has embedding opportunities.

- Use the **4-8 QnT bandwidth** to guide **safe embedding**.

- Modify coefficients **gently**:
    - **LSB modification**  
    - Minimal **±1 tweak**

- Keep track of **how many bits** we’ve embedded per image.

- Prep for **reconstruction** in the next step (**inverse DCT**).


### Code Block #1: The Embed Function (Revised and Cleaned Up)


In [ ]:
def embed_bits_in_dct_blocks_8x8(dct_blocks_info, embedding_candidates, payload_bits, low_threshold=4, high_threshold=8):
    """
    Embed bits into DCT coefficients of 8x8 blocks.

    Args:
        dct_blocks_info (list): DCT blocks for a single image.
        embedding_candidates (list): List of embedding spots per block.
        payload_bits (numpy array): Bitstream to embed.
        low_threshold (int): Minimum Q value allowed.
        high_threshold (int): Maximum Q value allowed.

    Returns:
        modified_dct_blocks_info (list): DCT blocks with embedded payload.
        num_embedded_bits (int): Number of bits actually embedded.
    """
    bit_idx = 0
    max_bits = len(payload_bits)
    import copy
    modified_dct_blocks_info = copy.deepcopy(dct_blocks_info)

    print(f"\nEmbedding up to {max_bits} bits in current image...")
    print(f"\n[Embedding Step] Total candidate blocks: {len(embedding_candidates)}")
    skipped_blocks = 0

    for block_candidates in embedding_candidates:
        if not block_candidates:
            continue

        block_position = block_candidates[0]['block_position']
        print(f"  Block: {block_position} has {len(block_candidates)} candidate frequencies")
        
        # Create a mapping from block positions to indices
        block_pos_to_index = {block["block_position"]: idx for idx, block in enumerate(dct_blocks_info)}

        block_idx = block_pos_to_index.get(block_position)
        if block_idx is None:
            print(f"    ⚠️ Block position {block_position} not found!")
            continue

        # Retrieve the DCT block corresponding to the current block position
        dct_block = modified_dct_blocks_info[block_idx]["dct_block"]

        for candidate in block_candidates:
            y, x = candidate['freq']
            q_value = candidate['q_value']

            if not (low_threshold <= q_value <= high_threshold):
                print(f"    Skipping freq ({y},{x}) with Q={q_value}")
                skipped_blocks += 1
                continue

            print(f"    ✅ Embedding bit at freq ({y},{x}), Q={q_value}")

            if bit_idx >= max_bits:
                print("All payload bits embedded!")
                return modified_dct_blocks_info, bit_idx

            for channel in range(3):  # R, G, B
                coeff = int(round(dct_block[y, x, channel]))
                if coeff >= 0:
                    new_coeff = (coeff & ~1) | payload_bits[bit_idx]
                else:
                    new_coeff = -((abs(coeff) & ~1) | payload_bits[bit_idx])

                dct_block[y, x, channel] = float(new_coeff)

                bit_idx += 1

                if bit_idx >= max_bits:
                    break

        if bit_idx >= max_bits:
            break

    print(f"Finished embedding {bit_idx} bits in this image.")
    return modified_dct_blocks_info, bit_idx



### Code Block #2: Loop Through All Images and Embed Payloads

In [ ]:
import copy

# Store modified DCT blocks after embedding
modified_dct_blocks_per_image = []
embedded_bits_per_image = []

for idx in range(len(filtered_candidates_per_image)):
    image_title = filtered_titles[idx]
    payload_bits = filtered_payloads[idx]
    num_payload_bits = len(payload_bits)
    num_candidate_blocks = len(filtered_candidates_per_image[idx])
    
    # Estimate capacity based on actual candidate frequency slots
    capacity_estimate = sum([len(block_candidates) for block_candidates in filtered_candidates_per_image[idx]])

    try:
        message_payload_str = ''.join([chr(int("".join(map(str, payload_bits[i:i+8])), 2)) for i in range(0, len(payload_bits), 8)])
    except:
        message_payload_str = "[Not a valid string - likely random bits]"

    print("\n" + "=" * 60)
    print(f"=== Embedding in Image {idx+1}: {image_title} ===")
    print("=" * 60)
    print(f"Original Payload String: {message_payload_str}")
    print(f"Converted Payload Bits: {''.join(map(str, payload_bits.tolist()))}")
    print(f"Number of Bits in Payload: {num_payload_bits}")
    print(f"Available Candidate Blocks: {num_candidate_blocks}")
    print(f"Estimated Capacity (sum of candidate frequencies): {capacity_estimate} bits")
    print(f"Using Bandwidth Threshold: Low=4, High=8")
    print("=" * 60)

    # Initialize bit index
    bit_idx = 0
    max_bits = len(payload_bits)

    # Deep copy the DCT blocks to modify them
    modified_dct_blocks = copy.deepcopy(filtered_dct_blocks_per_image[idx])

    for block_candidates in filtered_candidates_per_image[idx]:
        block_position = block_candidates[0]['block_position']
        
        # Create a mapping from block positions to indices
        block_pos_to_index = {block["block_position"]: idx for idx, block in enumerate(filtered_dct_blocks_per_image[idx])}

        block_idx = block_pos_to_index.get(block_position)
        if block_idx is None:
            print(f"    ⚠️ Block position {block_position} not found!")
            continue

        # Retrieve the DCT block corresponding to the current block position
        dct_block = modified_dct_blocks[block_idx]["dct_block"]

        for candidate in block_candidates:
            y, x = candidate['freq']
            q_value = candidate['q_value']

            if bit_idx >= max_bits:
                break  # Stop if all bits are embedded

            # Embed the bit #CHECK IG THHIS IS OKAY
            for channel in range(3):  # R, G, B
                coeff = int(round(dct_block[y, x, channel]))
                if abs(coeff) > 254:
                    print(f"⚠️ Skipping image '{filtered_titles[idx]}' → block {block_position}, freq ({y},{x}), channel {channel} — coeff too large: {coeff}")
                    continue
                if coeff >= 0:
                    new_coeff = (coeff & ~1) | payload_bits[bit_idx]
                else:
                    new_coeff = -((abs(coeff) & ~1) | payload_bits[bit_idx])
                    dct_block[y, x, channel] = float(new_coeff)
            bit_idx += 1

        if bit_idx >= max_bits:
            break  # Stop if all bits are embedded

    modified_dct_blocks_per_image.append(modified_dct_blocks)
    embedded_bits_per_image.append(bit_idx)

    print(f"\nDone Embedding in Image {idx+1}: {image_title}")
    print(f"Total Bits Embedded: {bit_idx}/{num_payload_bits} bits")
    print("=" * 60 + "\n")


### Code Block #3: Visual Check / Summary of Embedding

In [ ]:
# Summary of embedding per image
with redirect_stdout(log_buffer):
    print(f"\n--- Summary of Embedding [Batch {BATCH_INDEX+1}] ---")
    for idx, num_bits in enumerate(embedded_bits_per_image):
        print(f"Image {idx+1}: {filtered_titles[idx]} - {num_bits} bits embedded.")

## Step 10: Inverse DCT and Reconstruct the Stego Images

### Code Block #1: Helper Function to Rebuild 16x16 Blocks from Modified DCT Blocks

In [ ]:
def dct_block_to_spatial_block_8x8(dct_block):
    """
    Perform inverse DCT on a single 8x8x3 DCT block.

    Args:
        dct_block (numpy array): 8x8x3 DCT block.

    Returns:
        spatial_block (numpy array): 8x8x3 spatial block (uint8).
    """
    spatial_block = np.zeros_like(dct_block, dtype=np.float32)

    for c in range(3):  # For each RGB channel
        spatial_block[:, :, c] = cv2.idct(dct_block[:, :, c])

    # Debug check for out-of-bound values BEFORE clipping
    if spatial_block.max() > 255 or spatial_block.min() < 0:
        print(f"[WARN] IDCT result out of range: min={spatial_block.min()}, max={spatial_block.max()}")
    # Clip values and convert to uint8 for image display
    spatial_block = np.clip(spatial_block, 0, 255).astype(np.uint8)
    return spatial_block


### Code Block #2: Reconstruct the Entire Image From Modified Blocks

In [ ]:
def reconstruct_image_from_dct_blocks_8x8(original_image, modified_dct_blocks_info, block_size=8):
    """
    Reconstruct the stego image from modified 8x8 DCT blocks.

    Args:
        original_image (numpy array): The original RGB image (uint8).
        modified_dct_blocks_info (list): List of modified DCT blocks (with 'block_position' and 'dct_block').
        block_size (int): Block size (default 8).

    Returns:
        reconstructed_image (numpy array): The stego image reconstructed from DCT blocks.
    """
    reconstructed_image = original_image.copy()

    for block_info in modified_dct_blocks_info:
        block_position = block_info['block_position']
        dct_block = block_info['dct_block']

        # Inverse DCT to get spatial block
        spatial_block = dct_block_to_spatial_block_8x8(dct_block)

        # clip values to ensure they are in the valid range
        spatial_block = np.clip(spatial_block, 0, 255).astype(np.uint8)

        # Map it back to the image
        y_start = block_position[0] * block_size
        x_start = block_position[1] * block_size

        reconstructed_image[y_start:y_start+block_size, x_start:x_start+block_size, :] = spatial_block

    return reconstructed_image


### Code Block #3: Reconstruct and Display Stego Images Side by Side

In [ ]:
def display_original_and_stego_images_8x8(filtered_sample_images, reconstructed_images, filtered_titles):
    """
    Display original and stego images side by side for visual comparison.

    Args:
        filtered_sample_images (list): List of original RGB images (normalized or uint8).
        reconstructed_images (list): List of reconstructed stego RGB images (uint8).
        filtered_titles (list): List of titles for each image.
    """
    num_images = len(filtered_sample_images)
    fig, axes = plt.subplots(2, num_images, figsize=(num_images * 4, 8))

    if num_images == 1:
        axes = np.array(axes).reshape(2, 1)

    for i in range(num_images):
        # Normalize if needed
        original_img = (filtered_sample_images[i] * 255).astype(np.uint8) if filtered_sample_images[i].max() <= 1.0 else filtered_sample_images[i]
        stego_img = reconstructed_images[i]

        axes[0, i].imshow(original_img)
        axes[0, i].set_title(f"Original: {filtered_titles[i]}")
        axes[0, i].axis('off')

        axes[1, i].imshow(stego_img)
        axes[1, i].set_title(f"Stego: {filtered_titles[i]}")
        axes[1, i].axis('off')

    plt.tight_layout()
    plt.show()


### Code Block #4: Reconstruct Images and Run the Display

In [ ]:
def load_filtered_images_8x8(filtered_image_paths, target_size=(224, 224)):
    """
    Load and preprocess filtered images from file paths.

    Args:
        filtered_image_paths (list): List of image file paths.
        target_size (tuple): Resize dimensions, default (224, 224).

    Returns:
        filtered_sample_images (list): List of normalized RGB images.
    """
    filtered_sample_images = []

    for path in filtered_image_paths:
        img = Image.open(path).convert("RGB")
        img = img.resize(target_size)
        img_array = np.array(img) / 255.0  # Normalize to [0,1]
        filtered_sample_images.append(img_array)

    print(f"✅ Loaded and preprocessed {len(filtered_sample_images)} images at size {target_size}")
    return filtered_sample_images

filtered_sample_images = load_filtered_images_8x8(filtered_sample_image_paths)


In [ ]:
reconstructed_images = []

for idx in range(len(filtered_dct_blocks_per_image)):
    print(f"Reconstructing Image {idx+1}: {filtered_titles[idx]}")

    # Get original image (uint8) from normalized if needed
    original_img = (filtered_sample_images[idx] * 255).astype(np.uint8) if filtered_sample_images[idx].max() <= 1.0 else filtered_sample_images[idx]

    # Reconstruct stego image using 8x8 block-based function
    stego_img = reconstruct_image_from_dct_blocks_8x8(
        original_image=original_img,
        modified_dct_blocks_info=modified_dct_blocks_per_image[idx]
    )

    reconstructed_images.append(stego_img)

# Display side-by-side comparison
#display_original_and_stego_images_8x8(filtered_sample_images, reconstructed_images, filtered_titles)


## Step 11: Evaluate Image Quality - PSNR & SSIM & MSE

### Code Block #1: Setup PSNR & SSIM Functions

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import numpy as np

def evaluate_stego_images(filtered_sample_images, reconstructed_images, filtered_titles):
    """
    Evaluate PSNR, SSIM, and MSE between original and stego images.

    Args:
        filtered_sample_images (list): List of original images (normalized RGB arrays).
        reconstructed_images (list): List of reconstructed stego images (uint8).
        filtered_titles (list): Titles for each image.

    Returns:
        evaluation_results (list): List of dicts with PSNR, SSIM, and MSE per image.
    """
    evaluation_results = []

    for idx in range(len(filtered_sample_images)):
        # Prepare original and stego images in uint8
        original_img = (filtered_sample_images[idx] * 255).astype(np.uint8) if filtered_sample_images[idx].max() <= 1.0 else filtered_sample_images[idx]
        stego_img = reconstructed_images[idx]

        # PSNR (computed on full RGB images)
        psnr_value = psnr(original_img, stego_img, data_range=255)

        # SSIM (computed on full RGB images)
        ssim_value = ssim(
            original_img,
            stego_img,
            channel_axis=-1,   # Instead of multichannel=True
            data_range=255
        )

        # MSE (Mean Squared Error)
        mse_value = np.mean((original_img.astype("float32") - stego_img.astype("float32")) ** 2)

        result = {
            "Image": filtered_titles[idx],
            "PSNR (dB)": round(psnr_value, 2),
            "SSIM": round(ssim_value, 4),
            "MSE": round(mse_value, 4)
        }

        evaluation_results.append(result)

        print(f"Evaluated {filtered_titles[idx]} --> PSNR: {result['PSNR (dB)']} dB, SSIM: {result['SSIM']}, MSE: {result['MSE']}")

    return evaluation_results


### Code Block #2: Run Evaluation on Your Images

In [ ]:
evaluation_results = evaluate_stego_images(filtered_sample_images, reconstructed_images, filtered_titles)


### Code Block #3: Visualize The Scores


In [ ]:
import pandas as pd

def display_evaluation_results(evaluation_results):
    """
    Display the evaluation results in a table.

    Args:
        evaluation_results (list): List of dicts with evaluation data.
    """
    df = pd.DataFrame(evaluation_results)
    display(df)

df_eval = pd.DataFrame(evaluation_results)

# ✅ Display in notebook (unchanged)
display(df_eval)

# ✅ Save to plain-text log file
eval_table_log_path = os.path.join(LOG_DIR, "evaluation_table_log.txt")
with open(eval_table_log_path, "a") as logf:
    logf.write(f"\n=== [Batch {BATCH_INDEX + 1}] Evaluation Table (Plain Text) ===\n")
    logf.write(df_eval.to_string(index=False))
    logf.write("\n")

# ✅ Save to CSV for structured tracking
csv_path = os.path.join(LOG_DIR, f"evaluation_table_batch_{BATCH_INDEX + 1:02d}.csv")
df_eval.to_csv(csv_path, index=False)
print(f"[INFO] Evaluation table saved to:\n- TXT: {eval_table_log_path}\n- CSV: {csv_path}")

###   Code Block: Save Stego Images (PNG or JPEG)

In [ ]:
def save_original_and_stego_images(filtered_sample_images, reconstructed_images, filtered_titles, save_dir="/home/btm0050/Research_AI_Stegno_Analysis/SteganoGan/mount/100bit", format="jpeg", jpeg_quality=90):
    """
    Save both original and stego images to a directory in the specified format.

    Args:
        filtered_sample_images (list): List of original images (normalized RGB arrays).
        reconstructed_images (list): List of stego images (uint8).
        filtered_titles (list): Image titles to use as filenames.
        save_dir (str): Directory to save images.
        format (str): Image format ('png' for lossless, 'jpeg' for lossy).
        jpeg_quality (int): Quality for JPEG images (1-100).
    """
    import os
    from PIL import Image

    originals_dir = os.path.join(save_dir, "orig")
    stegos_dir = os.path.join(save_dir, "stego")

    # Create directories if they don't exist
    os.makedirs(originals_dir, exist_ok=True)
    os.makedirs(stegos_dir, exist_ok=True)

    for idx, (original_img, stego_img) in enumerate(zip(filtered_sample_images, reconstructed_images)):
        # Prepare original image (convert back to uint8 if needed)
        original_img_uint8 = (original_img * 255).astype(np.uint8) if original_img.max() <= 1.0 else original_img

        # Convert to PIL Images
        orig_pil = Image.fromarray(original_img_uint8)
        stego_pil = Image.fromarray(stego_img)

        # Get base filename (strip file extension)
        base_name = os.path.splitext(filtered_titles[idx])[0]

        # File paths
        orig_file = os.path.join(originals_dir, f"{base_name}_original.{format}")
        stego_file = os.path.join(stegos_dir, f"{base_name}_stego.{format}")

        # Save original
        if format.lower() == "jpeg":
            orig_pil.save(orig_file, "JPEG", quality=jpeg_quality)
        else:
            orig_pil.save(orig_file)

        # Save stego
        if format.lower() == "jpeg":
            stego_pil.save(stego_file, "JPEG", quality=jpeg_quality)
        else:
            stego_pil.save(stego_file)

        print(f"Saved Original Image: {orig_file}")
        print(f"Saved Stego Image: {stego_file}")

# Run it - save PNGs (or JPEGs if you like)
save_original_and_stego_images(filtered_sample_images, reconstructed_images, filtered_titles, save_dir="/home/btm0050/Research_AI_Stegno_Analysis/SteganoGan/mount/100bit", format="jpeg")


#print(f"✅ Batch complete. Next BATCH_INDEX = {BATCH_INDEX}")  # or "jpeg"


###Loop Through Dataset

In [ ]:
from IPython import get_ipython
import sys
from IPython.display import clear_output

# === ✅ Safety Check ===
if 'all_image_paths' not in globals():
    raise RuntimeError("❌ 'all_image_paths' is not defined. Please run setup cells (1–6) first.")

# === ⚙️ Batch Config ===
TOTAL_IMAGES = len(all_image_paths)
BATCH_SIZE = 1000
TOTAL_BATCHES = TOTAL_IMAGES // BATCH_SIZE
START_CELL_INDEX = 2  # Run from 7th cell onward (0-based index, includes markdown)

# === 🔁 Resume Support ===
progress_file = "last_completed_batch.txt"

log_dir = "./100bitlogs"
os.makedirs(log_dir, exist_ok=True)
log_file = os.path.join(log_dir, "batch_progress.txt")

try:
    with open(progress_file, "r") as f:
        content = f.read().strip()
        start_batch = int(content) + 1 if content else 0
except FileNotFoundError:
    start_batch = 0

print(f"🚀 Starting batch processing from batch {start_batch + 1} of {TOTAL_BATCHES}")

# === 🧠 Input Cell History ===
input_history = get_ipython().user_ns['_ih']

print("\n🔍 DEBUG: Showing first few lines of START_CELL_INDEX cell:")
print(input_history[START_CELL_INDEX][:300])  # Preview first 300 chars of the cell
# === 🔂 Batch Loop ===
for BATCH_INDEX in range(start_batch, TOTAL_BATCHES):
    print(f"\n🔁 Processing Batch {BATCH_INDEX + 1}/{TOTAL_BATCHES}")

    get_ipython().user_ns['BATCH_INDEX'] = BATCH_INDEX

    for idx, cell_code in enumerate(input_history[START_CELL_INDEX:-1], start=START_CELL_INDEX):
        if isinstance(cell_code, str):
            print(f"\n▶️ Executing cell #{idx + 1}...")
            try:
                exec(cell_code)
            except Exception as e:
                print(f"\n❌ Error in cell #{idx + 1} during Batch {BATCH_INDEX + 1}:")
                print(e)
                raise

    # Save progress
    with open(progress_file, "w") as f:
        f.write(str(BATCH_INDEX))

    # ✅ Log progress to file
    log_line = f"✅ Batch {BATCH_INDEX + 1} complete. Next will be {BATCH_INDEX + 2}"
    print(log_line)

    with open(log_file, "a", encoding="utf-8") as f:
        f.write(log_line + "\n")

    sys.stdout.flush()


# Clear all outputs in the notebook for every iteration

# Call this at the end of each batch iteration
clear_output(wait=True)


